In [ ]:
import json
import os
import shutil
import subprocess
import sys
import time
import zipfile
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

started = time.perf_counter()
configured = os.environ.get("KNEE_BUNDLE_DIR")
if configured:
    bundle = Path(configured).expanduser().resolve()
else:
    input_root = Path("/kaggle/input")
    manifests = (
        sorted(
            {
                path
                for pattern in (
                    "*/manifest.json",
                    "*/*/manifest.json",
                    "*/*/*/manifest.json",
                )
                for path in input_root.glob(pattern)
            }
        )
        if input_root.is_dir()
        else []
    )
    archives = (
        sorted(
            {
                path
                for pattern in ("*/bundle.zip", "*/*/bundle.zip", "*/*/*/bundle.zip")
                for path in input_root.glob(pattern)
            }
        )
        if input_root.is_dir()
        else []
    )
    if len(manifests) == 1:
        bundle = manifests[0].parent
    elif len(archives) == 1:
        bundle = Path("/kaggle/working/knee_bundle")
        if bundle.exists():
            shutil.rmtree(bundle)
        bundle.mkdir(parents=True)
        with zipfile.ZipFile(archives[0]) as archive:
            archive.extractall(bundle)
    else:
        raise RuntimeError(
            f"Expected one mounted bundle, manifests={manifests}, archives={archives}"
        )
wheels = sorted((bundle / "wheels").glob("*.whl"))
if wheels:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--no-index",
            "--no-deps",
            *map(str, wheels),
        ],
        check=True,
    )
manifest = json.loads((bundle / "manifest.json").read_text())
versions = {}
for item in manifest["dependencies"]:
    try:
        versions[item["name"]] = version(item["name"])
    except PackageNotFoundError:
        versions[item["name"]] = None
print(
    json.dumps(
        {
            "bundle": str(bundle),
            "dependency_versions": versions,
            "setup_seconds": time.perf_counter() - started,
        }
    )
)

In [ ]:
data_root = os.environ.get("KNEE_DATA_ROOT")
output = (
    Path(
        os.environ.get(
            "KNEE_OUTPUT",
            "/kaggle/working/submission.csv"
            if Path("/kaggle/working").is_dir()
            else "submission.csv",
        )
    )
    .expanduser()
    .resolve()
)
command = [
    sys.executable,
    "-m",
    "scripts.pipeline.predict",
    str(bundle / "config.json"),
    "--bundle",
    str(bundle),
    "--output",
    str(output),
    "--device",
    os.environ.get("KNEE_DEVICE", "auto"),
    "--precision",
    os.environ.get("KNEE_PRECISION", "auto"),
    "--batch-size",
    os.environ.get("KNEE_BATCH_SIZE", str(manifest["runtime"]["batch_size"])),
    "--num-workers",
    os.environ.get("KNEE_NUM_WORKERS", str(manifest["runtime"]["num_workers"])),
]
if data_root:
    command.extend(["--data-root", data_root])
environment = os.environ.copy()
environment["PYTHONPATH"] = str(bundle / "code")
run_started = time.perf_counter()
subprocess.run(command, check=True, env=environment)
print(
    json.dumps(
        {
            "inference_command_seconds": time.perf_counter() - run_started,
            "output": str(output),
        }
    )
)

In [ ]:
sys.path.insert(0, str(bundle / "code"))
import csv
import math

from knee_abnormality.data.sources import resolve_raw_dir
from knee_abnormality.inference.bundle import discover_kaggle_data_root

root = (
    Path(data_root).expanduser().resolve() if data_root else discover_kaggle_data_root()
)
template_path = resolve_raw_dir(root, "test") / "sample_submission.csv"
with template_path.open(newline="") as stream:
    template = list(csv.reader(stream))
with output.open(newline="") as stream:
    submission = list(csv.reader(stream))
if not submission or submission[0] != template[0] or len(submission) != len(template):
    raise RuntimeError(
        "Submission schema or row count differs from the mounted template"
    )
if [row[0] for row in submission[1:]] != [row[0] for row in template[1:]]:
    raise RuntimeError("Submission study order differs from the mounted template")
for row in submission[1:]:
    values = [float(value) for value in row[1:]]
    if not all(math.isfinite(value) and 0 <= value <= 1 for value in values):
        raise RuntimeError(f"Invalid probabilities for study {row[0]}")
print(
    json.dumps(
        {
            "validated_rows": len(submission) - 1,
            "columns": submission[0],
            "total_seconds": time.perf_counter() - started,
        }
    )
)